# Intro to sqlite3 and Querying Databases from Python

While files like CSV and JSON store raw data, databases store structured and relational data - the kind we use for analytics, lookups, and joins.

 As a data engineer, you'll frequently need to:
 
- Run SQL queries from Python notebooks
- Read/write tables from pipelines
- Automate ingestion jobs using SQL + Python logic

sqlite3 is the perfect lightweight database to learn this bridge - no setup, runs entirely in memory or local file, and uses standard SQL syntax.

### What Is SQLite?

SQLite is a self-contained, file-based SQL database engine that comes preinstalled with Python.

 It's great for:

- Practicing SQL queries
- Prototyping ETL jobs
- Storing intermediate results locally

No server, no configuration - just Python and a file!

# Initial setup of SQLLite 

In [0]:
import sqlite3

# Step 1: Connect to (or create) the database
conn = sqlite3.connect("retail.db")
cursor = conn.cursor()


# Step 2: Create tables
cursor.executescript("""
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS order_items;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_name TEXT NOT NULL,
    city TEXT,
    signup_date TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL,
    category TEXT,
    price REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    order_date TEXT,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    item_total REAL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

# Step 3: Insert sample data
cursor.executescript("""
INSERT INTO customers (customer_name, city, signup_date) VALUES
('Akash', 'Mumbai', '2023-01-12'),
('Priya', 'Delhi', '2023-02-20'),
('Rohan', 'Pune', '2023-03-05'),
('Neha', 'Bangalore', '2023-04-10'),
('Raj', 'Hyderabad', '2023-05-01');

INSERT INTO products (product_name, category, price) VALUES
('Laptop', 'Electronics', 70000),
('Smartphone', 'Electronics', 25000),
('Headphones', 'Accessories', 2000),
('T-shirt', 'Apparel', 800),
('Shoes', 'Footwear', 3000);

INSERT INTO orders (customer_id, order_date, total_amount) VALUES
(1, '2023-05-01', 72000),
(2, '2023-05-03', 27000),
(3, '2023-05-05', 3800),
(1, '2023-05-10', 800),
(4, '2023-05-12', 3200);

INSERT INTO order_items (order_id, product_id, quantity, item_total) VALUES
(1, 1, 1, 70000),
(1, 3, 1, 2000),
(2, 2, 1, 25000),
(2, 3, 1, 2000),
(3, 4, 2, 1600),
(3, 5, 1, 2200),
(4, 4, 1, 800),
(5, 5, 1, 3200);
""")

# Step 4: Commit and close
conn.commit()
conn.close()

print(" retail.db created successfully with sample data!")



### 5. Reading Data (SELECT Queries)

Once data is inserted, you can query it just like SQL.

In [0]:
import sqlite3, pandas as pd

conn = sqlite3.connect("retail.db")

df = pd.read_sql("SELECT * FROM customers;", conn)
print(df)

conn.close()

### 6. Filtering and Conditions

Just like SQL, you can use WHERE and ORDER BY.

In [0]:
import sqlite3, pandas as pd

conn = sqlite3.connect("retail.db")

df = pd.read_sql("SELECT * FROM customers where city = 'Mumbai';", conn)
print(df)

conn.close()

### 7. Updating and Deleting Records

In [0]:
import sqlite3, pandas as pd

conn = sqlite3.connect("retail.db")
cursor = conn.cursor()



cursor.executescript("update customers set city = 'Mumbai11' where customer_id = 1;")

df = pd.read_sql("SELECT * FROM customers where customer_id = 1;", conn)
conn.commit()
print(df)

conn.close()

### Updating Values

In [0]:
import sqlite3, pandas as pd

conn = sqlite3.connect("retail.db")
cursor = conn.cursor()

df = pd.read_sql("SELECT * FROM orders order by total_amount", conn)
conn.commit()
print(df)

conn.close()

In [0]:
import sqlite3, pandas as pd

conn = sqlite3.connect("retail.db")
cursor = conn.cursor()


cursor.executescript("delete from orders where total_amount > 70000;")

df = pd.read_sql("SELECT * FROM orders order by total_amount;", conn)
conn.commit()
print(df)

conn.close()

## Reading Results as Dictionaries

By default, SQLite returns tuples.
 Let's configure it to return rows as dictionary-like objects.

In [0]:
conn = sqlite3.connect("retail.db")
conn.row_factory = sqlite3.Row 
cursor = conn.cursor()

cursor.execute("SELECT * FROM orders")
for row in cursor.fetchall():
    print(dict(row))

### closing the connection

In [0]:
conn.close()
print(" Connection closed successfully!")

## Python + SQL = the backbone of every data pipeline, ingestion, and transformation job.

##Practice Tasks

- Create a table products with columns (id, name, price).
- Insert 5 sample records using executemany().
- Query all products with price > 1000.
- Update a record and verify the change.
- Fetch all results as dictionaries.